# Data Preparation

## Pilot Study Data Preparation

In [1]:
import os
import glob
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import random

# Define paths relative to the notebook workspace
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
MIXED_DIR = "../data/mixed"

# Create directories
for inst in ["baglama", "zurna"]:
    os.makedirs(os.path.join(PROCESSED_DIR, inst), exist_ok=True)
os.makedirs(MIXED_DIR, exist_ok=True)

print("Directories setup complete.")

Directories setup complete.


In [ ]:
SR = 22050 # Standard sample rate
CHUNK_LENGTH = 5 # seconds
SAMPLES_PER_CHUNK = SR * CHUNK_LENGTH

def time_to_sec(t_str):
    if pd.isna(t_str): return None
    t_str = str(t_str).strip()
    parts = t_str.split(':')
    if len(parts) == 2:
        return int(parts[0]) * 60 + int(parts[1])
    elif len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    return 0

# Load dataset index
index_df = pd.read_csv(os.path.join(RAW_DIR, "dataset_index.csv"))
index_df['start_sec'] = index_df['start_time'].apply(time_to_sec)
index_df['end_sec'] = index_df['end_time'].apply(time_to_sec)

# 1. Process Zurna files
zurna_files = glob.glob(os.path.join(RAW_DIR, "zurna", "*.wav"))
zurna_chunks = []

print(f"Processing {len(zurna_files)} Zurna files...")
for f in zurna_files:
    file_basename = os.path.basename(f)
    # Extract youtube_id assuming format like youtube_id-Zurna.wav or just youtube_id
    youtube_id = file_basename.split('-')[0].replace('.wav', '')

    audio, sr = librosa.load(f, sr=SR)

    # Check if there are specific segments for this file in the dataset index
    segments = index_df[(index_df['youtube_id'] == youtube_id) & (index_df['instrument'] == 'zurna')]

    if len(segments) > 0:
        # Concatenate valid segments
        valid_audio = []
        for _, row in segments.iterrows():
            start_sample = int(row['start_sec'] * SR) if pd.notnull(row['start_sec']) else 0
            end_sample = int(row['end_sec'] * SR) if pd.notnull(row['end_sec']) else len(audio)
            valid_audio.append(audio[start_sample:end_sample])
        audio = np.concatenate(valid_audio) if valid_audio else audio

    if len(audio) < SAMPLES_PER_CHUNK:
        audio = np.pad(audio, (0, SAMPLES_PER_CHUNK - len(audio)))

    # Chunking
    num_chunks = len(audio) // SAMPLES_PER_CHUNK
    for i in range(num_chunks):
        chunk = audio[i*SAMPLES_PER_CHUNK : (i+1)*SAMPLES_PER_CHUNK]
        chunk_name = f"zurna_{file_basename.replace('.wav', '')}_{i}.wav"
        chunk_path = os.path.join(PROCESSED_DIR, "zurna", chunk_name)
        sf.write(chunk_path, chunk, SR)
        zurna_chunks.append(chunk_path)

total_zurna_duration = len(zurna_chunks) * CHUNK_LENGTH
print(f"Total Zurna chunks: {len(zurna_chunks)} ({total_zurna_duration} seconds)")

Processing 2 Zurna files...


Total Zurna chunks: 60 (300 seconds)


In [ ]:
# 2. Process Baglama files to match Zurna duration
baglama_files = glob.glob(os.path.join(RAW_DIR, "baglama", "*.wav"))
random.shuffle(baglama_files)

baglama_chunks = []
baglama_duration = 0

print(f"Processing Baglama files to match {total_zurna_duration} seconds...")
for f in baglama_files:
    if baglama_duration >= total_zurna_duration:
        break

    file_basename = os.path.basename(f)
    youtube_id = file_basename.split('-')[0].replace('.wav', '')

    audio, sr = librosa.load(f, sr=SR)

    # Check if there are specific segments for this file in the dataset index
    segments = index_df[(index_df['youtube_id'] == youtube_id) & (index_df['instrument'] == 'baglama')]

    if len(segments) > 0:
        valid_audio = []
        for _, row in segments.iterrows():
            start_sample = int(row['start_sec'] * SR) if pd.notnull(row['start_sec']) else 0
            end_sample = int(row['end_sec'] * SR) if pd.notnull(row['end_sec']) else len(audio)
            valid_audio.append(audio[start_sample:end_sample])
        audio = np.concatenate(valid_audio) if valid_audio else audio

    num_chunks = len(audio) // SAMPLES_PER_CHUNK

    for i in range(num_chunks):
        if baglama_duration >= total_zurna_duration:
            break

        chunk = audio[i*SAMPLES_PER_CHUNK : (i+1)*SAMPLES_PER_CHUNK]
        chunk_name = f"baglama_{file_basename.replace('.wav', '')}_{i}.wav"
        chunk_path = os.path.join(PROCESSED_DIR, "baglama", chunk_name)
        sf.write(chunk_path, chunk, SR)
        baglama_chunks.append(chunk_path)
        baglama_duration += CHUNK_LENGTH

print(f"Total Baglama chunks: {len(baglama_chunks)} ({baglama_duration} seconds)")

Processing Baglama files to match 300 seconds...


Total Baglama chunks: 60 (300 seconds)


In [4]:
# 3. Mix the chunks
print("Mixing chunks...")
metadata = []

# Pair them up
pairs = list(zip(zurna_chunks, baglama_chunks))

for idx, (z_path, b_path) in enumerate(pairs):
    z_audio, _ = librosa.load(z_path, sr=SR)
    b_audio, _ = librosa.load(b_path, sr=SR)

    # Mix them: element-wise addition
    mixed = z_audio + b_audio

    # Normalize to prevent clipping
    max_val = np.max(np.abs(mixed))
    if max_val > 0:
        mixed = mixed / max_val

    mix_name = f"mixed_{idx:03d}.wav"
    mix_path = os.path.join(MIXED_DIR, mix_name)
    sf.write(mix_path, mixed, SR)

    metadata.append({
        "mix_file": mix_name,
        "zurna_source": os.path.basename(z_path),
        "baglama_source": os.path.basename(b_path),
        "duration": CHUNK_LENGTH
    })

# Save metadata
df = pd.DataFrame(metadata)
df.to_csv(os.path.join(MIXED_DIR, "metadata.csv"), index=False)
print(f"Mixed {len(pairs)} files. Metadata saved to {MIXED_DIR}/metadata.csv")

Mixing chunks...


Mixed 60 files. Metadata saved to ../data/mixed/metadata.csv
